# 04 -- Multi-Query Batch Evaluation

When optimizing a retrieval pipeline you need to evaluate across many queries, not just one. `BatchRankFlow` aggregates metrics, identifies failure cases, and provides diagnostic visualizations across a query set.

In [ ]:
%matplotlib inline

import numpy as np

from rankflow import RankFlow, BatchRankFlow

## Generating synthetic multi-query data

We simulate 30 queries, each with 20 documents passing through a 3-step pipeline. For each query, 3 documents are randomly marked as relevant.

In [ ]:
rng = np.random.default_rng(42)
n_queries = 30
n_docs = 20
n_steps = 3
step_labels = ["BM25", "Semantic", "Cross-Encoder"]
chunk_labels = [f"doc_{i}" for i in range(n_docs)]

rankflows = []
for q in range(n_queries):
    # Each step is a slightly perturbed permutation
    base = rng.permutation(n_docs)
    ranks = np.array([
        base,
        np.argsort(np.argsort(base + rng.normal(0, 3, n_docs))),
        np.argsort(np.argsort(base + rng.normal(0, 5, n_docs))),
    ])
    # Pick 3 random relevant docs
    rel = rng.choice(chunk_labels, size=3, replace=False).tolist()

    rf = RankFlow(
        ranks=ranks,
        step_labels=step_labels,
        chunk_labels=chunk_labels,
        relevant_chunks=rel,
    )
    rf.query_label = f"query_{q}"
    rankflows.append(rf)

batch = BatchRankFlow(rankflows)
print(f"BatchRankFlow with {len(batch.rankflows)} queries")

## Aggregated metrics

`aggregate_metrics(k)` computes mean and standard deviation across all queries for every metric at every step.

In [ ]:
agg = batch.aggregate_metrics(k=5)

print("Per-step aggregated metrics:")
for i, step in enumerate(agg["per_step"]):
    print(f"  {step_labels[i]}: NDCG@5={step['ndcg_at_k_mean']:.3f} +/- {step['ndcg_at_k_std']:.3f}")

## Box plot per step

In [ ]:
batch.plot(k=5, metric="ndcg_at_k")

## Metric evolution with error bars

In [ ]:
batch.plot_metric_evolution(k=5)

## Metrics dashboard

`plot_dashboard(k)` shows all five metrics as box plots in a single figure.

In [ ]:
batch.plot_dashboard(k=5)

## Win / loss / tie analysis

For each step transition, count how many queries improved (win), degraded (loss), or stayed the same (tie).

In [ ]:
for wl in batch.win_loss_analysis(metric="ndcg_at_k", k=5):
    print(f"{wl['transition']}: {wl['wins']}W / {wl['losses']}L / {wl['ties']}T  "
          f"(win rate: {wl['win_pct']:.0f}%)")

## Segmentation by query difficulty

Splits queries into difficulty buckets (hard, medium, easy) based on their final-step metric value, then shows how each bucket evolves across steps.

In [ ]:
batch.plot_by_difficulty(k=5, metric="ndcg_at_k", buckets=3)

## Improvement heatmap

Rows are queries (sorted by total improvement), columns are step transitions, color intensity shows the metric delta.

In [ ]:
batch.plot_improvement_heatmap(metric="ndcg_at_k", k=5)

## Failure case identification

Find queries where the pipeline made things worse (final metric significantly below initial).

In [ ]:
failures = batch.failure_cases(metric="ndcg_at_k", k=5, threshold=-0.05)
print(f"Found {len(failures)} failure cases:")
for f in failures[:5]:
    print(f"  {f['query_label']}: {f['initial_value']:.3f} -> {f['final_value']:.3f} (delta={f['delta']:+.3f})")

You can then drill into a specific failure case to see what happened:

In [ ]:
if failures:
    idx = failures[0]["query_index"]
    batch.rankflows[idx].plot()

---

**Next:** [05 -- Adapters and Export](05_adapters_and_export.ipynb) covers importing from TREC/RAGAS/ranx formats, exporting results, and modeling hybrid pipelines with MergeRankFlow.